In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import os

if os.path.exists("/kaggle/working/best_densenet169_multiclass.pkl"):
    print("Model already exists — skipping training")
    SKIP_TRAINING = True
else:
    SKIP_TRAINING = False

In [ ]:
if not SKIP_TRAINING:
    main()

# 3-class dataset preparation/merging


In [ ]:
# =============================================================================
#  CELL 1 — MERGE ALL 3 DATASETS
#  Paths updated to match your exact Kaggle dataset structure
# =============================================================================

import os, shutil, zipfile, random
from pathlib import Path
import pandas as pd
from tqdm import tqdm
from PIL import Image, ImageEnhance, ImageFilter
# ── EXACT PATHS from your Kaggle dataset panel ────────────────────────────────
ORAL_CANCER_DIR = "/kaggle/input/datasets/zaidpy/oral-cancer-dataset/Oral cancer Dataset 2.0/OC Dataset kaggle new"
HISTO_DIR       = "/kaggle/input/datasets/ashenafifasilkebede/dataset"
NDB_DIR         = "/kaggle/input/datasets/nitinkmath/nbd-ufes"
NDB_IMAGES_DIR  = f"{NDB_DIR}/images"
NDB_EXCEL       = f"{NDB_DIR}/ndb-ufes.xlsx"

OUTPUT_DIR      = "/kaggle/working/merged_dataset"
IMG_EXTS        = {".jpg", ".jpeg", ".png", ".bmp", ".tiff"}

# ── Create output folders ─────────────────────────────────────────────────────
classes  = ["Normal", "OSCC", "Leukoplakia"]
counters = {cls: 0 for cls in classes}

for cls in classes:
    Path(f"{OUTPUT_DIR}/{cls}").mkdir(parents=True, exist_ok=True)

def copy_img(src: Path, target_class: str):
    counters[target_class] += 1
    new_name = f"{target_class}_{counters[target_class]:06d}{src.suffix.lower()}"
    shutil.copy2(src, Path(OUTPUT_DIR) / target_class / new_name)

# ─────────────────────────────────────────────────────────────────────────────
#  DATASET 1 — Oral Cancer Dataset
#  CANCER → OSCC  |  NON CANCER → Normal
# ─────────────────────────────────────────────────────────────────────────────
print("=" * 55)
print("  DATASET 1: Oral Cancer Dataset")
print("=" * 55)

for src_folder, target_cls in [("CANCER", "OSCC"), ("NON CANCER", "Normal")]:
    folder = Path(ORAL_CANCER_DIR) / src_folder
    if not folder.exists():
        print(f"  ⚠ Not found: {folder}")
        continue
    imgs = [f for f in folder.iterdir() if f.suffix.lower() in IMG_EXTS]
    for img in tqdm(imgs, desc=f"  {src_folder} → {target_cls}", leave=False):
        copy_img(img, target_cls)
    print(f"  ✔ {src_folder:<15} → {target_cls:<12}: {len(imgs):>4} images")

# ─────────────────────────────────────────────────────────────────────────────
#  DATASET 2 — Histopathological Imaging Dataset
#  Has train / val / test splits — use ALL of them
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 55)
print("  DATASET 2: Histopathological Imaging Dataset")
print("=" * 55)

HISTO_MAP = {
    "normal": "Normal", "Normal": "Normal", "NORMAL": "Normal",
    "oscc":   "OSCC",   "OSCC":   "OSCC",
}

for split in ["train", "val", "test"]:
    split_dir = Path(HISTO_DIR) / split
    if not split_dir.exists():
        continue
    for sub in split_dir.iterdir():
        if not sub.is_dir(): continue
        target_cls = HISTO_MAP.get(sub.name)
        if not target_cls: continue
        imgs = [f for f in sub.rglob("*") if f.suffix.lower() in IMG_EXTS]
        for img in tqdm(imgs, desc=f"  {split}/{sub.name} → {target_cls}", leave=False):
            copy_img(img, target_cls)
        print(f"  ✔ {split}/{sub.name:<10} → {target_cls:<12}: {len(imgs):>4} images")

# ─────────────────────────────────────────────────────────────────────────────
#  DATASET 3 — NDB-UFES (236 images)
#  Uses ndb-ufes.xlsx diagnosis column for labels
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 55)
print("  DATASET 3: NDB-UFES")
print("=" * 55)

df_ndb = pd.read_excel(NDB_EXCEL)
print(f"  Rows: {len(df_ndb)}")
print(f"  Diagnosis counts:")
print(df_ndb["diagnosis"].value_counts().to_string())
print()

NDB_MAP = {
    "OSCC":                          "OSCC",
    "Leukoplakia with dysplasia":    "Leukoplakia",
    "Leukoplakia without dysplasia": "Leukoplakia",
    "Leukoplakia":                   "Leukoplakia",
    "Normal":                        "Normal",
    "normal":                        "Normal",
}

copied = skipped = 0
img_folder = Path(NDB_IMAGES_DIR)

for _, row in tqdm(df_ndb.iterrows(), total=len(df_ndb),
                   desc="  Copying NDB images", leave=False):
    img_name  = str(row["path"]).strip()
    diagnosis = str(row["diagnosis"]).strip()
    target    = NDB_MAP.get(diagnosis)

    if target is None:
        for k, v in NDB_MAP.items():
            if k.lower() in diagnosis.lower():
                target = v
                break

    if target is None:
        skipped += 1
        continue

    img_file = img_folder / img_name
    if not img_file.exists():
        results  = list(img_folder.rglob(img_name))
        img_file = results[0] if results else None

    if img_file and Path(img_file).exists():
        copy_img(Path(img_file), target)
        copied += 1
    else:
        skipped += 1

print(f"  ✔ Copied  : {copied}")
print(f"  ⚠ Skipped : {skipped}")

# ─────────────────────────────────────────────────────────────────────────────
#  MERGED SUMMARY
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 55)
print("  MERGED DATASET — COUNTS (before balancing)")
print("=" * 55)
total = 0
for cls in classes:
    n   = len(list((Path(OUTPUT_DIR) / cls).iterdir()))
    bar = "█" * (n // 30)
    print(f"  {cls:<15}: {n:>5,}  {bar}")
    total += n
print(f"  {'─'*45}")
print(f"  {'TOTAL':<15}: {total:>5,}")
print("=" * 55)
print("\n  ✔ Merge complete — now running balance + augmentation ...")



In [ ]:

# =============================================================================
#  CELL 2 — BALANCE & AUGMENT TO 4000 PER CLASS
#  Target: 4000 per class = 12,000 total
#  10 augmentation recipes for maximum variety
# =============================================================================

BALANCED_DIR = "/kaggle/working/balanced_dataset"
TARGET       = 4000

# ── 10 augmentation recipes ───────────────────────────────────────────────────
def augment(img: Image.Image, recipe: int) -> Image.Image:
    r = recipe % 10

    if r == 0:
        img = img.transpose(Image.FLIP_LEFT_RIGHT)
        img = ImageEnhance.Brightness(img).enhance(random.uniform(0.70, 1.40))
    elif r == 1:
        img = img.rotate(random.choice([90, 180, 270]))
        img = ImageEnhance.Contrast(img).enhance(random.uniform(0.70, 1.50))
    elif r == 2:
        img = img.transpose(Image.FLIP_TOP_BOTTOM)
        img = ImageEnhance.Color(img).enhance(random.uniform(0.60, 1.60))
    elif r == 3:
        img = img.rotate(random.uniform(-35, 35), expand=False)
        img = ImageEnhance.Sharpness(img).enhance(random.uniform(0.4, 2.2))
    elif r == 4:
        w, h = img.size
        cx, cy = random.uniform(0.05, 0.20), random.uniform(0.05, 0.20)
        img = img.crop((int(w*cx), int(h*cy), int(w*(1-cx)), int(h*(1-cy))))
        img = img.resize((224, 224), Image.BILINEAR)
        img = ImageEnhance.Brightness(img).enhance(random.uniform(0.85, 1.25))
    elif r == 5:
        img = img.filter(ImageFilter.GaussianBlur(radius=random.uniform(0.5, 2.0)))
        img = img.transpose(Image.FLIP_LEFT_RIGHT)
        img = ImageEnhance.Contrast(img).enhance(random.uniform(0.85, 1.40))
    elif r == 6:
        img = img.rotate(random.uniform(-20, 20), expand=False)
        img = img.transpose(Image.FLIP_LEFT_RIGHT)
        img = ImageEnhance.Color(img).enhance(random.uniform(0.75, 1.45))
        img = ImageEnhance.Brightness(img).enhance(random.uniform(0.80, 1.30))
    elif r == 7:
        w, h = img.size
        pad  = int(min(w, h) * 0.15)
        img  = img.crop((pad, pad, w-pad, h-pad))
        img  = img.resize((224, 224), Image.BILINEAR)
        img  = img.rotate(random.uniform(-15, 15), expand=False)
    elif r == 8:
        img = ImageEnhance.Color(img).enhance(random.uniform(0.5, 2.0))
        img = img.rotate(random.uniform(-25, 25), expand=False)
        img = ImageEnhance.Brightness(img).enhance(random.uniform(0.80, 1.25))
    else:
        w, h  = img.size
        scale = random.uniform(0.70, 0.92)
        nw, nh = int(w*scale), int(h*scale)
        left  = (w - nw) // 2
        top   = (h - nh) // 2
        img   = img.crop((left, top, left+nw, top+nh))
        img   = img.resize((224, 224), Image.LANCZOS)
        img   = ImageEnhance.Sharpness(img).enhance(random.uniform(0.6, 1.8))

    return img.convert("RGB")

# ── Create balanced folders & copy originals ──────────────────────────────────
print("\n" + "=" * 55)
print(f"  AUGMENTING ALL CLASSES TO {TARGET} IMAGES EACH")
print("=" * 55)

for cls in classes:
    Path(f"{BALANCED_DIR}/{cls}").mkdir(parents=True, exist_ok=True)

print("\n  Step 1: Copying originals ...")
for cls in classes:
    src_dir = Path(OUTPUT_DIR) / cls
    imgs    = [f for f in src_dir.iterdir() if f.suffix.lower() in IMG_EXTS]
    for img in imgs:
        shutil.copy2(img, Path(BALANCED_DIR) / cls / img.name)
    print(f"  ✔ {cls:<15}: {len(imgs):>5,} originals copied")

# ── Augment each class up to TARGET ──────────────────────────────────────────
print(f"\n  Step 2: Augmenting to {TARGET} per class ...")

for cls in classes:
    src_imgs = [f for f in (Path(OUTPUT_DIR) / cls).iterdir()
                if f.suffix.lower() in IMG_EXTS]
    current  = len(list((Path(BALANCED_DIR) / cls).iterdir()))
    needed   = TARGET - current

    if needed <= 0:
        print(f"  {cls:<15}: already at {current:,} — no augmentation needed")
        continue

    print(f"\n  {cls}: {current:,} → {TARGET:,}  (need {needed:,} more)")

    aug_count = 0
    recipe    = 0
    pbar      = tqdm(total=needed, desc=f"  {cls}", leave=True, unit="img")

    while aug_count < needed:
        src = src_imgs[aug_count % len(src_imgs)]
        try:
            pil  = Image.open(src).convert("RGB")
            aug  = augment(pil, recipe)
            name = f"{cls}_aug_{aug_count:07d}.jpg"
            aug.save(Path(BALANCED_DIR) / cls / name, "JPEG", quality=93)
        except Exception:
            pass
        aug_count += 1
        recipe    += 1
        pbar.update(1)
    pbar.close()

# ── Final summary ─────────────────────────────────────────────────────────────
print("\n" + "=" * 55)
print("  ✅  FINAL BALANCED DATASET")
print("=" * 55)
grand = 0
for cls in classes:
    n   = len(list((Path(BALANCED_DIR) / cls).iterdir()))
    bar = "█" * (n // 200)
    print(f"  {cls:<15}: {n:>6,}  {bar}")
    grand += n
print(f"  {'─'*45}")
print(f"  {'TOTAL':<15}: {grand:>6,}")
print("=" * 55)
print(f"\n  ✔ Saved → {BALANCED_DIR}")
print(f"  3 classes × {TARGET} images = {3*TARGET:,} total ✔")

DENSENET for 3-class

In [ ]:
# Cell 3 — Fix NumPy then run training
!pip install "numpy<2" -q

In [ ]:
# Cell 1
!pip install torch==2.2.0+cu118 torchvision==0.17.0+cu118 \
    --index-url https://download.pytorch.org/whl/cu118 -q
!pip install "numpy<2" open-clip-torch transformers -q

In [ ]:
# Cell 2 — restart kernel then verify
import torch
torch.backends.cudnn.enabled = True
x = torch.randn(2,3,224,224).cuda()
print(f"✔ GPU ready: {torch.cuda.get_device_name(0)}")

In [ ]:
# =============================================================================
#  merge_and_balance_multiclass.py
#  Merges 5 datasets → 4 or 5 class balanced dataset
#  Run as a Kaggle notebook cell
#
#  Sources:
#   1. Your existing oral cancer dataset  (CANCER / NON CANCER)
#   2. Kaggle histopathology dataset       (Normal / OSCC)
#   3. NDB-UFES dataset                    (OSCC / Leukoplakia)
#   4. Mohamedgobara dataset               (Normal / Benign / OPMD / Cancer)
#   5. Roboflow leukoplakia dataset        (leukoplakia images)
#
#  Output classes (4 or 5 depending on what's available):
#   → Normal, OSCC, Leukoplakia, Benign, Erythroplakia (if found)
# =============================================================================

import os, shutil, random
from pathlib import Path
import pandas as pd
from tqdm import tqdm
from PIL import Image, ImageEnhance, ImageFilter

# ─────────────────────────────────────────────────────────────────────────────
#  STEP 0 — CHECK EXACT FOLDER NAMES IN YOUR DATASETS
#  Run this cell first to see what's available before merging
# ─────────────────────────────────────────────────────────────────────────────

def inspect_datasets():
    datasets = {
        "Oral Cancer (yours)":    "/kaggle/input/datasets/zaidpy/oral-cancer-dataset/Oral cancer Dataset 2.0/OC Dataset kaggle new",
        "Histopathology":         "/kaggle/input/datasets/ashenafifasilkebede/dataset",
        "NDB-UFES":               "/kaggle/input/datasets/nitinkmath/nbd-ufes",
        "Mohamedgobara":          "/kaggle/input/datasets/mohamedgobara/oral-lesions-malignancy-detection-dataset/Oral Images Dataset",
        "Roboflow Leukoplakia":   "/kaggle/input/datasets/nitinkmath/leukoplakia",
    }

    print("=" * 60)
    print("  DATASET INSPECTION")
    print("=" * 60)

    for name, path in datasets.items():
        p = Path(path)
        print(f"\n▸ {name}")
        print(f"  Path: {path}")
        if not p.exists():
            print("  ⚠ NOT FOUND — update path")
            continue
        # Print folder tree 2 levels deep
        for item in sorted(p.iterdir()):
            if item.is_dir():
                n_imgs = len([f for f in item.rglob("*")
                              if f.suffix.lower() in {".jpg",".jpeg",".png",".bmp"}])
                print(f"  📁 {item.name}/  ({n_imgs} images)")
                for sub in sorted(item.iterdir()):
                    if sub.is_dir():
                        n = len([f for f in sub.rglob("*")
                                 if f.suffix.lower() in {".jpg",".jpeg",".png",".bmp"}])
                        print(f"       📁 {sub.name}/  ({n} images)")

# ── Run inspection first ──────────────────────────────────────────────────────
inspect_datasets()


# ─────────────────────────────────────────────────────────────────────────────
#  STEP 1 — PATHS
#  ⚠ Update these paths after running inspect_datasets() above
# ─────────────────────────────────────────────────────────────────────────────

# Source datasets
ORAL_CANCER_DIR = "/kaggle/input/datasets/zaidpy/oral-cancer-dataset/Oral cancer Dataset 2.0/OC Dataset kaggle new"
HISTO_DIR       = "/kaggle/input/datasets/ashenafifasilkebede/dataset"
NDB_DIR         = "/kaggle/input/datasets/nitinkmath/nbd-ufes"
NDB_IMAGES_DIR  = f"{NDB_DIR}/images"
NDB_EXCEL       = f"{NDB_DIR}/ndb-ufes.xlsx"
GOBARA_DIR      = "/kaggle/input/datasets/mohamedgobara/oral-lesions-malignancy-detection-dataset/Oral Images Dataset"
ROBOFLOW_DIR    = "/kaggle/input/datasets/nitinkmath/leukoplakia"   # adjust to your upload name

# Output
MERGED_DIR   = "/kaggle/working/merged_multiclass"
BALANCED_DIR = "/kaggle/working/balanced_multiclass"
TARGET       = 3000   # images per class after balancing
IMG_EXTS     = {".jpg", ".jpeg", ".png", ".bmp", ".tiff"}

# ─────────────────────────────────────────────────────────────────────────────
#  STEP 2 — LABEL MAPPINGS
#
#  ⚠ IMPORTANT: Update GOBARA_MAP after checking inspect_datasets() output
#  The mohamedgobara dataset folder names might be:
#  "Normal", "Benign", "OPMD", "Oral_Cancer" or similar
#  Update the keys below to match exactly what you see
# ─────────────────────────────────────────────────────────────────────────────

# Your existing dataset
ORAL_CANCER_MAP = {
    "CANCER":     "OSCC",
    "NON CANCER": "Normal",
}

# Kaggle histopathology (train/val/test splits, Normal/OSCC subfolders)
HISTO_MAP = {
    "Normal": "Normal",
    "normal": "Normal",
    "NORMAL": "Normal",
    "OSCC":   "OSCC",
    "oscc":   "OSCC",
}

# NDB-UFES Excel diagnosis column
NDB_MAP = {
    "OSCC":                          "OSCC",
    "Leukoplakia with dysplasia":    "Leukoplakia",
    "Leukoplakia without dysplasia": "Leukoplakia",
    "Leukoplakia":                   "Leukoplakia",
    "Normal":                        "Normal",
    "normal":                        "Normal",
}

# Mohamedgobara dataset
# ⚠ UPDATE KEYS after checking folder names with inspect_datasets()
GOBARA_MAP = {
    "Normal":     "Normal",
    "normal":     "Normal",
    "Benign":     "Benign",
    "benign":     "Benign",
    "OPMD":       "Leukoplakia",   # OPMD maps to Leukoplakia
    "opmd":       "Leukoplakia",
    "Oral_Cancer":"OSCC",
    "oral_cancer":"OSCC",
    "Cancer":     "OSCC",
    "cancer":     "OSCC",
    # If erythroplakia exists as its own folder:
    "Erythroplakia": "Erythroplakia",
    "erythroplakia": "Erythroplakia",
}

# Roboflow — all images in train/images/ and valid/images/ are Leukoplakia
ROBOFLOW_CLASS = "Leukoplakia"


# ─────────────────────────────────────────────────────────────────────────────
#  STEP 3 — MERGE ALL DATASETS
# ─────────────────────────────────────────────────────────────────────────────

# Determine classes dynamically — will add Erythroplakia if found
CLASSES  = ["Normal", "OSCC", "Leukoplakia", "Benign"]
counters = {}

def ensure_class(cls_name):
    if cls_name not in CLASSES:
        CLASSES.append(cls_name)
    if cls_name not in counters:
        counters[cls_name] = 0
    Path(MERGED_DIR) / cls_name
    (Path(MERGED_DIR) / cls_name).mkdir(parents=True, exist_ok=True)

def copy_img(src: Path, target_class: str):
    ensure_class(target_class)
    counters[target_class] = counters.get(target_class, 0) + 1
    new_name = f"{target_class}_{counters[target_class]:06d}{src.suffix.lower()}"
    dst = Path(MERGED_DIR) / target_class / new_name
    shutil.copy2(src, dst)

# Create base output folders
for cls in CLASSES:
    (Path(MERGED_DIR) / cls).mkdir(parents=True, exist_ok=True)
    counters[cls] = 0

# ── Dataset 1: Oral Cancer ────────────────────────────────────────────────────
print("=" * 55)
print("  DATASET 1: Oral Cancer Dataset")
print("=" * 55)
for src_name, target_cls in ORAL_CANCER_MAP.items():
    folder = Path(ORAL_CANCER_DIR) / src_name
    if not folder.exists():
        print(f"  ⚠ Not found: {folder}")
        continue
    imgs = [f for f in folder.iterdir() if f.suffix.lower() in IMG_EXTS]
    for img in tqdm(imgs, desc=f"  {src_name} → {target_cls}", leave=False):
        copy_img(img, target_cls)
    print(f"  ✔ {src_name:<20} → {target_cls:<15}: {len(imgs):>5}")

# ── Dataset 2: Histopathology ─────────────────────────────────────────────────
print("\n" + "=" * 55)
print("  DATASET 2: Histopathology")
print("=" * 55)
for split in ["train", "val", "test"]:
    split_dir = Path(HISTO_DIR) / split
    if not split_dir.exists():
        continue
    for sub in split_dir.iterdir():
        if not sub.is_dir(): continue
        target_cls = HISTO_MAP.get(sub.name)
        if not target_cls: continue
        imgs = [f for f in sub.rglob("*") if f.suffix.lower() in IMG_EXTS]
        for img in tqdm(imgs, desc=f"  {split}/{sub.name}", leave=False):
            copy_img(img, target_cls)
        print(f"  ✔ {split}/{sub.name:<12} → {target_cls:<15}: {len(imgs):>5}")

# ── Dataset 3: NDB-UFES ───────────────────────────────────────────────────────
print("\n" + "=" * 55)
print("  DATASET 3: NDB-UFES")
print("=" * 55)
if Path(NDB_EXCEL).exists():
    df_ndb = pd.read_excel(NDB_EXCEL)
    img_folder = Path(NDB_IMAGES_DIR)
    copied = skipped = 0
    for _, row in tqdm(df_ndb.iterrows(), total=len(df_ndb),
                       desc="  NDB-UFES", leave=False):
        img_name  = str(row["path"]).strip()
        diagnosis = str(row["diagnosis"]).strip()
        target    = NDB_MAP.get(diagnosis)
        if target is None:
            for k, v in NDB_MAP.items():
                if k.lower() in diagnosis.lower():
                    target = v; break
        if target is None:
            skipped += 1; continue
        img_file = img_folder / img_name
        if not img_file.exists():
            results  = list(img_folder.rglob(img_name))
            img_file = results[0] if results else None
        if img_file and Path(img_file).exists():
            copy_img(Path(img_file), target)
            copied += 1
        else:
            skipped += 1
    print(f"  ✔ Copied: {copied}  |  Skipped: {skipped}")
else:
    print(f"  ⚠ NDB Excel not found at {NDB_EXCEL}")

# ── Dataset 4: Mohamedgobara ──────────────────────────────────────────────────
print("\n" + "=" * 55)
print("  DATASET 4: Mohamedgobara Oral Lesions")
print("=" * 55)
gobara_root = Path(GOBARA_DIR)
if gobara_root.exists():
    for folder in sorted(gobara_root.rglob("*")):
        if not folder.is_dir(): continue
        target_cls = GOBARA_MAP.get(folder.name)
        if not target_cls: continue
        imgs = [f for f in folder.iterdir() if f.suffix.lower() in IMG_EXTS]
        if not imgs: continue
        for img in tqdm(imgs, desc=f"  {folder.name} → {target_cls}", leave=False):
            copy_img(img, target_cls)
        print(f"  ✔ {folder.name:<20} → {target_cls:<15}: {len(imgs):>5}")
else:
    print(f"  ⚠ Not found: {GOBARA_DIR}")

# ── Dataset 5: Roboflow Leukoplakia ──────────────────────────────────────────
print("\n" + "=" * 55)
print("  DATASET 5: Roboflow Leukoplakia")
print("=" * 55)
roboflow_root = Path(ROBOFLOW_DIR)
if roboflow_root.exists():
    # Images are in train/images/ and valid/images/
    for split in ["train", "valid", "test"]:
        img_dir = roboflow_root / split / "images"
        if not img_dir.exists():
            # Try flat structure
            img_dir = roboflow_root / split
        if not img_dir.exists():
            continue
        imgs = [f for f in img_dir.iterdir() if f.suffix.lower() in IMG_EXTS]
        for img in tqdm(imgs, desc=f"  {split}/images → Leukoplakia", leave=False):
            copy_img(img, ROBOFLOW_CLASS)
        print(f"  ✔ {split}/images → Leukoplakia: {len(imgs):>5}")
else:
    print(f"  ⚠ Not found: {ROBOFLOW_DIR}")

# ── Merged Summary ────────────────────────────────────────────────────────────
print("\n" + "=" * 55)
print("  MERGED DATASET — BEFORE BALANCING")
print("=" * 55)
total = 0
final_classes = []
for cls in CLASSES:
    cls_dir = Path(MERGED_DIR) / cls
    if not cls_dir.exists():
        continue
    n   = len(list(cls_dir.iterdir()))
    if n == 0:
        continue
    final_classes.append(cls)
    bar = "█" * (n // 30)
    print(f"  {cls:<18}: {n:>6,}  {bar}")
    total += n
print(f"  {'─'*45}")
print(f"  {'TOTAL':<18}: {total:>6,}")
print("=" * 55)
print(f"\n  Classes found: {final_classes}")
print(f"  Running balance step now...\n")


# ─────────────────────────────────────────────────────────────────────────────
#  STEP 4 — AUGMENTATION RECIPES
# ─────────────────────────────────────────────────────────────────────────────

def augment(img: Image.Image, recipe: int) -> Image.Image:
    r = recipe % 10
    if r == 0:
        img = img.transpose(Image.FLIP_LEFT_RIGHT)
        img = ImageEnhance.Brightness(img).enhance(random.uniform(0.75, 1.40))
    elif r == 1:
        img = img.rotate(random.choice([90, 180, 270]))
        img = ImageEnhance.Contrast(img).enhance(random.uniform(0.75, 1.50))
    elif r == 2:
        img = img.transpose(Image.FLIP_TOP_BOTTOM)
        img = ImageEnhance.Color(img).enhance(random.uniform(0.65, 1.55))
    elif r == 3:
        img = img.rotate(random.uniform(-35, 35), expand=False)
        img = ImageEnhance.Sharpness(img).enhance(random.uniform(0.4, 2.2))
    elif r == 4:
        w, h = img.size
        cx, cy = random.uniform(0.05, 0.20), random.uniform(0.05, 0.20)
        img = img.crop((int(w*cx), int(h*cy), int(w*(1-cx)), int(h*(1-cy))))
        img = img.resize((224, 224), Image.BILINEAR)
        img = ImageEnhance.Brightness(img).enhance(random.uniform(0.88, 1.25))
    elif r == 5:
        img = img.filter(ImageFilter.GaussianBlur(radius=random.uniform(0.5, 2.0)))
        img = img.transpose(Image.FLIP_LEFT_RIGHT)
        img = ImageEnhance.Contrast(img).enhance(random.uniform(0.85, 1.40))
    elif r == 6:
        img = img.rotate(random.uniform(-20, 20), expand=False)
        img = img.transpose(Image.FLIP_LEFT_RIGHT)
        img = ImageEnhance.Color(img).enhance(random.uniform(0.75, 1.45))
        img = ImageEnhance.Brightness(img).enhance(random.uniform(0.80, 1.30))
    elif r == 7:
        w, h = img.size
        pad  = int(min(w, h) * 0.15)
        img  = img.crop((pad, pad, w-pad, h-pad))
        img  = img.resize((224, 224), Image.BILINEAR)
        img  = img.rotate(random.uniform(-15, 15), expand=False)
    elif r == 8:
        img = ImageEnhance.Color(img).enhance(random.uniform(0.5, 2.0))
        img = img.rotate(random.uniform(-25, 25), expand=False)
        img = ImageEnhance.Brightness(img).enhance(random.uniform(0.80, 1.25))
    else:
        w, h  = img.size
        scale = random.uniform(0.70, 0.92)
        nw, nh = int(w*scale), int(h*scale)
        left, top = (w-nw)//2, (h-nh)//2
        img = img.crop((left, top, left+nw, top+nh))
        img = img.resize((224, 224), Image.LANCZOS)
        img = ImageEnhance.Sharpness(img).enhance(random.uniform(0.6, 1.8))
    return img.convert("RGB")


# ─────────────────────────────────────────────────────────────────────────────
#  STEP 5 — BALANCE TO TARGET PER CLASS
# ─────────────────────────────────────────────────────────────────────────────

print("=" * 55)
print(f"  BALANCING ALL CLASSES TO {TARGET} IMAGES EACH")
print("=" * 55)

for cls in final_classes:
    (Path(BALANCED_DIR) / cls).mkdir(parents=True, exist_ok=True)

# Copy originals
print("\n  Step 1: Copying originals ...")
for cls in final_classes:
    src_dir = Path(MERGED_DIR) / cls
    imgs    = [f for f in src_dir.iterdir() if f.suffix.lower() in IMG_EXTS]
    for img in imgs:
        shutil.copy2(img, Path(BALANCED_DIR) / cls / img.name)
    print(f"  ✔ {cls:<18}: {len(imgs):>6,} originals")

# Augment minority classes
print(f"\n  Step 2: Augmenting to {TARGET} per class ...")
for cls in final_classes:
    src_imgs = [f for f in (Path(MERGED_DIR) / cls).iterdir()
                if f.suffix.lower() in IMG_EXTS]
    current  = len(list((Path(BALANCED_DIR) / cls).iterdir()))
    needed   = TARGET - current

    if needed <= 0:
        print(f"  {cls:<18}: already at {current:,} — no augmentation")
        continue

    print(f"\n  {cls}: {current:,} → {TARGET:,}  (need {needed:,} more)")
    aug_count = recipe = 0
    pbar = tqdm(total=needed, desc=f"  Augmenting {cls}", leave=True, unit="img")

    while aug_count < needed:
        src = src_imgs[aug_count % len(src_imgs)]
        try:
            pil  = Image.open(src).convert("RGB")
            aug  = augment(pil, recipe)
            name = f"{cls}_aug_{aug_count:07d}.jpg"
            aug.save(Path(BALANCED_DIR) / cls / name, "JPEG", quality=92)
        except Exception:
            pass
        aug_count += 1
        recipe    += 1
        pbar.update(1)
    pbar.close()

# ── Final summary ─────────────────────────────────────────────────────────────
print("\n" + "=" * 55)
print("  ✅  FINAL BALANCED DATASET")
print("=" * 55)
grand = 0
for cls in final_classes:
    n   = len(list((Path(BALANCED_DIR) / cls).iterdir()))
    bar = "█" * (n // 150)
    print(f"  {cls:<18}: {n:>6,}  {bar}")
    grand += n
print(f"  {'─'*45}")
print(f"  {'TOTAL':<18}: {grand:>6,}")
print("=" * 55)
print(f"\n  Classes : {final_classes}")
print(f"  Target  : {TARGET} per class")
print(f"  Saved → {BALANCED_DIR}")
print(f"\n  ✔ Ready for multiclass fine-tuning!")

In [ ]:
# Run this to see gobara folder structure
from pathlib import Path

gobara = Path("/kaggle/input/datasets/mohamedgobara/oral-lesions-malignancy-detection-dataset/Oral Images Dataset")
for p in sorted(gobara.rglob("*")):
    if p.is_dir():
        n = len([f for f in p.iterdir() 
                 if f.suffix.lower() in {".jpg",".jpeg",".png"}])
        depth = len(p.relative_to(gobara).parts)
        print("  " * depth + f"📁 {p.name}/  ({n} imgs)")

# 4 class dataset 

In [ ]:
# =============================================================================
#  merge_and_balance_multiclass.py
#  Merges 5 datasets → 4 or 5 class balanced dataset
#  Run as a Kaggle notebook cell
#
#  Sources:
#   1. Your existing oral cancer dataset  (CANCER / NON CANCER)
#   2. Kaggle histopathology dataset       (Normal / OSCC)
#   3. NDB-UFES dataset                    (OSCC / Leukoplakia)
#   4. Mohamedgobara dataset              (Normal / Benign / OPMD / Cancer)
#   5. Roboflow leukoplakia dataset       (leukoplakia images)
#
#  Output classes (4 or 5 depending on what's available):
#   → Normal, OSCC, Leukoplakia, Benign, Erythroplakia (if found)
# =============================================================================

import os, shutil, random
from pathlib import Path
import pandas as pd
from tqdm import tqdm
from PIL import Image, ImageEnhance, ImageFilter

# ─────────────────────────────────────────────────────────────────────────────
#  STEP 0 — CHECK EXACT FOLDER NAMES IN YOUR DATASETS
#  Run this cell first to see what's available before merging
# ─────────────────────────────────────────────────────────────────────────────

def inspect_datasets():
    datasets = {
        "Oral Cancer (yours)":    "/kaggle/input/datasets/zaidpy/oral-cancer-dataset/Oral cancer Dataset 2.0/OC Dataset kaggle new",
        "Histopathology":         "/kaggle/input/datasets/ashenafifasilkebede/dataset",
        "NDB-UFES":               "/kaggle/input/datasets/nitinkmath/nbd-ufes",
        "Mohamedgobara":          "/kaggle/input/datasets/mohamedgobara/oral-lesions-malignancy-detection-dataset/Oral Images Dataset",
        "Roboflow Leukoplakia":   "/kaggle/input/datasets/nitinkmath/leukoplakia",
    }

    print("=" * 60)
    print("  DATASET INSPECTION")
    print("=" * 60)

    for name, path in datasets.items():
        p = Path(path)
        print(f"\n▸ {name}")
        print(f"  Path: {path}")
        if not p.exists():
            print("  ⚠ NOT FOUND — update path")
            continue
        # Print folder tree 2 levels deep
        for item in sorted(p.iterdir()):
            if item.is_dir():
                n_imgs = len([f for f in item.rglob("*")
                              if f.suffix.lower() in {".jpg",".jpeg",".png",".bmp"}])
                print(f"  📁 {item.name}/  ({n_imgs} images)")
                for sub in sorted(item.iterdir()):
                    if sub.is_dir():
                        n = len([f for f in sub.rglob("*")
                                 if f.suffix.lower() in {".jpg",".jpeg",".png",".bmp"}])
                        print(f"       📁 {sub.name}/  ({n} images)")

# ── Run inspection first ──────────────────────────────────────────────────────
inspect_datasets()


# ─────────────────────────────────────────────────────────────────────────────
#  STEP 1 — PATHS
#  ⚠ Update these paths after running inspect_datasets() above
# ─────────────────────────────────────────────────────────────────────────────

# Source datasets
ORAL_CANCER_DIR = "/kaggle/input/datasets/zaidpy/oral-cancer-dataset/Oral cancer Dataset 2.0/OC Dataset kaggle new"
HISTO_DIR       = "/kaggle/input/datasets/ashenafifasilkebede/dataset"
NDB_DIR         = "/kaggle/input/datasets/nitinkmath/nbd-ufes"
NDB_IMAGES_DIR  = f"{NDB_DIR}/images"
NDB_EXCEL       = f"{NDB_DIR}/ndb-ufes.xlsx"
GOBARA_DIR      = "/kaggle/input/datasets/mohamedgobara/oral-lesions-malignancy-detection-dataset/Oral Images Dataset"
ROBOFLOW_DIR    = "/kaggle/input/datasets/nitinkmath/leukoplakia"   # adjust to your upload name

# Output
MERGED_DIR   = "/kaggle/working/merged_multiclass_final"
BALANCED_DIR = "/kaggle/working/balanced_multiclass_final"
TARGET       = 3000   # images per class after balancing
IMG_EXTS     = {".jpg", ".jpeg", ".png", ".bmp", ".tiff"}

# ─────────────────────────────────────────────────────────────────────────────
#  STEP 2 — LABEL MAPPINGS
# ─────────────────────────────────────────────────────────────────────────────

# Your existing dataset
ORAL_CANCER_MAP = {
    "CANCER":     "OSCC",
    "NON CANCER": "Normal",
}

# Kaggle histopathology (train/val/test splits, Normal/OSCC subfolders)
HISTO_MAP = {
    "Normal": "Normal",
    "normal": "Normal",
    "NORMAL": "Normal",
    "OSCC":   "OSCC",
    "oscc":   "OSCC",
}

# NDB-UFES Excel diagnosis column
NDB_MAP = {
    "OSCC":                          "OSCC",
    "Leukoplakia with dysplasia":    "Leukoplakia",
    "Leukoplakia without dysplasia": "Leukoplakia",
    "Leukoplakia":                   "Leukoplakia",
    "Normal":                        "Normal",
    "normal":                        "Normal",
}

# Mohamedgobara dataset
# Updated to match your nested folder layout:
# augmented_data/augmented_benign, augmented_data/augmented_malignant,
# original_data/benign_lesions, original_data/malignant_lesions
GOBARA_MAP = {
    "augmented_benign":    "Benign",
    "benign_lesions":      "Benign",
    "augmented_malignant": "OSCC",
    "malignant_lesions":   "OSCC",

    # Keep these too in case any higher-level folders or direct folders exist
    "Normal":              "Normal",
    "normal":              "Normal",
    "Benign":              "Benign",
    "benign":              "Benign",
    "OPMD":                "Leukoplakia",
    "opmd":                "Leukoplakia",
    "Oral_Cancer":         "OSCC",
    "oral_cancer":         "OSCC",
    "Cancer":              "OSCC",
    "cancer":              "OSCC",
    "Erythroplakia":       "Erythroplakia",
    "erythroplakia":       "Erythroplakia",
}

# Roboflow — all images in train/images/ and valid/images/ are Leukoplakia
ROBOFLOW_CLASS = "Leukoplakia"


# ─────────────────────────────────────────────────────────────────────────────
#  STEP 3 — MERGE ALL DATASETS
# ─────────────────────────────────────────────────────────────────────────────

# Determine classes dynamically — will add Erythroplakia if found
CLASSES  = ["Normal", "OSCC", "Leukoplakia", "Benign"]
counters = {}

def ensure_class(cls_name):
    if cls_name not in CLASSES:
        CLASSES.append(cls_name)
    if cls_name not in counters:
        counters[cls_name] = 0
    Path(MERGED_DIR) / cls_name
    (Path(MERGED_DIR) / cls_name).mkdir(parents=True, exist_ok=True)

def copy_img(src: Path, target_class: str):
    ensure_class(target_class)
    counters[target_class] = counters.get(target_class, 0) + 1
    new_name = f"{target_class}_{counters[target_class]:06d}{src.suffix.lower()}"
    dst = Path(MERGED_DIR) / target_class / new_name
    shutil.copy2(src, dst)

# Create base output folders
for cls in CLASSES:
    (Path(MERGED_DIR) / cls).mkdir(parents=True, exist_ok=True)
    counters[cls] = 0

# ── Dataset 1: Oral Cancer ────────────────────────────────────────────────────
print("=" * 55)
print("  DATASET 1: Oral Cancer Dataset")
print("=" * 55)
for src_name, target_cls in ORAL_CANCER_MAP.items():
    folder = Path(ORAL_CANCER_DIR) / src_name
    if not folder.exists():
        print(f"  ⚠ Not found: {folder}")
        continue
    imgs = [f for f in folder.iterdir() if f.suffix.lower() in IMG_EXTS]
    for img in tqdm(imgs, desc=f"  {src_name} → {target_cls}", leave=False):
        copy_img(img, target_cls)
    print(f"  ✔ {src_name:<20} → {target_cls:<15}: {len(imgs):>5}")

# ── Dataset 2: Histopathology ─────────────────────────────────────────────────
print("\n" + "=" * 55)
print("  DATASET 2: Histopathology")
print("=" * 55)
for split in ["train", "val", "test"]:
    split_dir = Path(HISTO_DIR) / split
    if not split_dir.exists():
        continue
    for sub in split_dir.iterdir():
        if not sub.is_dir(): continue
        target_cls = HISTO_MAP.get(sub.name)
        if not target_cls: continue
        imgs = [f for f in sub.rglob("*") if f.suffix.lower() in IMG_EXTS]
        for img in tqdm(imgs, desc=f"  {split}/{sub.name}", leave=False):
            copy_img(img, target_cls)
        print(f"  ✔ {split}/{sub.name:<12} → {target_cls:<15}: {len(imgs):>5}")

# ── Dataset 3: NDB-UFES ───────────────────────────────────────────────────────
print("\n" + "=" * 55)
print("  DATASET 3: NDB-UFES")
print("=" * 55)
if Path(NDB_EXCEL).exists():
    df_ndb = pd.read_excel(NDB_EXCEL)
    img_folder = Path(NDB_IMAGES_DIR)
    copied = skipped = 0
    for _, row in tqdm(df_ndb.iterrows(), total=len(df_ndb),
                       desc="  NDB-UFES", leave=False):
        img_name  = str(row["path"]).strip()
        diagnosis = str(row["diagnosis"]).strip()
        target    = NDB_MAP.get(diagnosis)
        if target is None:
            for k, v in NDB_MAP.items():
                if k.lower() in diagnosis.lower():
                    target = v; break
        if target is None:
            skipped += 1; continue
        img_file = img_folder / img_name
        if not img_file.exists():
            results  = list(img_folder.rglob(img_name))
            img_file = results[0] if results else None
        if img_file and Path(img_file).exists():
            copy_img(Path(img_file), target)
            copied += 1
        else:
            skipped += 1
    print(f"  ✔ Copied: {copied}  |  Skipped: {skipped}")
else:
    print(f"  ⚠ NDB Excel not found at {NDB_EXCEL}")

# ── Dataset 4: Mohamedgobara ──────────────────────────────────────────────────
print("\n" + "=" * 55)
print("  DATASET 4: Mohamedgobara Oral Lesions")
print("=" * 55)

gobara_root = Path(GOBARA_DIR)
if gobara_root.exists():
    # Recursively scan everything under the provided Gobara directory.
    # This is needed because the dataset layout is nested:
    #   augmented_data/augmented_benign
    #   augmented_data/augmented_malignant
    #   original_data/benign_lesions
    #   original_data/malignant_lesions
    for folder in sorted([p for p in gobara_root.rglob("*") if p.is_dir()]):
        folder_name = folder.name.strip()

        target_cls = GOBARA_MAP.get(folder_name)

        # fallback keyword matching if exact key is not found
        if not target_cls:
            low = folder_name.lower()
            if "benign" in low:
                target_cls = "Benign"
            elif "malignant" in low or "cancer" in low or "oscc" in low:
                target_cls = "OSCC"
            elif "opmd" in low or "leuko" in low:
                target_cls = "Leukoplakia"
            elif "normal" in low:
                target_cls = "Normal"

        if not target_cls:
            continue

        imgs = [f for f in folder.rglob("*") if f.is_file() and f.suffix.lower() in IMG_EXTS]
        if not imgs:
            continue

        rel_name = str(folder.relative_to(gobara_root))
        for img in tqdm(imgs, desc=f"  {rel_name} → {target_cls}", leave=False):
            copy_img(img, target_cls)

        print(f"  ✔ {rel_name:<45} → {target_cls:<15}: {len(imgs):>5}")
else:
    print(f"  ⚠ Not found: {GOBARA_DIR}")

# ── Dataset 5: Roboflow Leukoplakia ──────────────────────────────────────────
print("\n" + "=" * 55)
print("  DATASET 5: Roboflow Leukoplakia")
print("=" * 55)
roboflow_root = Path(ROBOFLOW_DIR)
if roboflow_root.exists():
    # Images are in train/images/ and valid/images/
    for split in ["train", "valid", "test"]:
        img_dir = roboflow_root / split / "images"
        if not img_dir.exists():
            # Try flat structure
            img_dir = roboflow_root / split
        if not img_dir.exists():
            continue
        imgs = [f for f in img_dir.iterdir() if f.suffix.lower() in IMG_EXTS]
        for img in tqdm(imgs, desc=f"  {split}/images → Leukoplakia", leave=False):
            copy_img(img, ROBOFLOW_CLASS)
        print(f"  ✔ {split}/images → Leukoplakia: {len(imgs):>5}")
else:
    print(f"  ⚠ Not found: {ROBOFLOW_DIR}")

# ── Merged Summary ────────────────────────────────────────────────────────────
print("\n" + "=" * 55)
print("  MERGED DATASET — BEFORE BALANCING")
print("=" * 55)
total = 0
final_classes = []
for cls in CLASSES:
    cls_dir = Path(MERGED_DIR) / cls
    if not cls_dir.exists():
        continue
    n   = len(list(cls_dir.iterdir()))
    if n == 0:
        continue
    final_classes.append(cls)
    bar = "█" * (n // 30)
    print(f"  {cls:<18}: {n:>6,}  {bar}")
    total += n
print(f"  {'─'*45}")
print(f"  {'TOTAL':<18}: {total:>6,}")
print("=" * 55)
print(f"\n  Classes found: {final_classes}")
print(f"  Running balance step now...\n")


# ─────────────────────────────────────────────────────────────────────────────
#  STEP 4 — AUGMENTATION RECIPES
# ─────────────────────────────────────────────────────────────────────────────

def augment(img: Image.Image, recipe: int) -> Image.Image:
    r = recipe % 10
    if r == 0:
        img = img.transpose(Image.FLIP_LEFT_RIGHT)
        img = ImageEnhance.Brightness(img).enhance(random.uniform(0.75, 1.40))
    elif r == 1:
        img = img.rotate(random.choice([90, 180, 270]))
        img = ImageEnhance.Contrast(img).enhance(random.uniform(0.75, 1.50))
    elif r == 2:
        img = img.transpose(Image.FLIP_TOP_BOTTOM)
        img = ImageEnhance.Color(img).enhance(random.uniform(0.65, 1.55))
    elif r == 3:
        img = img.rotate(random.uniform(-35, 35), expand=False)
        img = ImageEnhance.Sharpness(img).enhance(random.uniform(0.4, 2.2))
    elif r == 4:
        w, h = img.size
        cx, cy = random.uniform(0.05, 0.20), random.uniform(0.05, 0.20)
        img = img.crop((int(w*cx), int(h*cy), int(w*(1-cx)), int(h*(1-cy))))
        img = img.resize((224, 224), Image.BILINEAR)
        img = ImageEnhance.Brightness(img).enhance(random.uniform(0.88, 1.25))
    elif r == 5:
        img = img.filter(ImageFilter.GaussianBlur(radius=random.uniform(0.5, 2.0)))
        img = img.transpose(Image.FLIP_LEFT_RIGHT)
        img = ImageEnhance.Contrast(img).enhance(random.uniform(0.85, 1.40))
    elif r == 6:
        img = img.rotate(random.uniform(-20, 20), expand=False)
        img = img.transpose(Image.FLIP_LEFT_RIGHT)
        img = ImageEnhance.Color(img).enhance(random.uniform(0.75, 1.45))
        img = ImageEnhance.Brightness(img).enhance(random.uniform(0.80, 1.30))
    elif r == 7:
        w, h = img.size
        pad  = int(min(w, h) * 0.15)
        img  = img.crop((pad, pad, w-pad, h-pad))
        img  = img.resize((224, 224), Image.BILINEAR)
        img  = img.rotate(random.uniform(-15, 15), expand=False)
    elif r == 8:
        img = ImageEnhance.Color(img).enhance(random.uniform(0.5, 2.0))
        img = img.rotate(random.uniform(-25, 25), expand=False)
        img = ImageEnhance.Brightness(img).enhance(random.uniform(0.80, 1.25))
    else:
        w, h  = img.size
        scale = random.uniform(0.70, 0.92)
        nw, nh = int(w*scale), int(h*scale)
        left, top = (w-nw)//2, (h-nh)//2
        img = img.crop((left, top, left+nw, top+nh))
        img = img.resize((224, 224), Image.LANCZOS)
        img = ImageEnhance.Sharpness(img).enhance(random.uniform(0.6, 1.8))
    return img.convert("RGB")


# ─────────────────────────────────────────────────────────────────────────────
#  STEP 5 — BALANCE TO TARGET PER CLASS
# ─────────────────────────────────────────────────────────────────────────────

print("=" * 55)
print(f"  BALANCING ALL CLASSES TO {TARGET} IMAGES EACH")
print("=" * 55)

for cls in final_classes:
    (Path(BALANCED_DIR) / cls).mkdir(parents=True, exist_ok=True)

# Copy originals
print("\n  Step 1: Copying originals ...")
for cls in final_classes:
    src_dir = Path(MERGED_DIR) / cls
    imgs    = [f for f in src_dir.iterdir() if f.suffix.lower() in IMG_EXTS]
    for img in imgs:
        shutil.copy2(img, Path(BALANCED_DIR) / cls / img.name)
    print(f"  ✔ {cls:<18}: {len(imgs):>6,} originals")

# Augment minority classes
print(f"\n  Step 2: Augmenting to {TARGET} per class ...")
for cls in final_classes:
    src_imgs = [f for f in (Path(MERGED_DIR) / cls).iterdir()
                if f.suffix.lower() in IMG_EXTS]
    current  = len(list((Path(BALANCED_DIR) / cls).iterdir()))
    needed   = TARGET - current

    if needed <= 0:
        print(f"  {cls:<18}: already at {current:,} — no augmentation")
        continue

    print(f"\n  {cls}: {current:,} → {TARGET:,}  (need {needed:,} more)")
    aug_count = recipe = 0
    pbar = tqdm(total=needed, desc=f"  Augmenting {cls}", leave=True, unit="img")

    while aug_count < needed:
        src = src_imgs[aug_count % len(src_imgs)]
        try:
            pil  = Image.open(src).convert("RGB")
            aug  = augment(pil, recipe)
            name = f"{cls}_aug_{aug_count:07d}.jpg"
            aug.save(Path(BALANCED_DIR) / cls / name, "JPEG", quality=92)
        except Exception:
            pass
        aug_count += 1
        recipe    += 1
        pbar.update(1)
    pbar.close()

# ── Final summary ─────────────────────────────────────────────────────────────
print("\n" + "=" * 55)
print("  ✅  FINAL BALANCED DATASET")
print("=" * 55)
grand = 0
for cls in final_classes:
    n   = len(list((Path(BALANCED_DIR) / cls).iterdir()))
    bar = "█" * (n // 150)
    print(f"  {cls:<18}: {n:>6,}  {bar}")
    grand += n
print(f"  {'─'*45}")
print(f"  {'TOTAL':<18}: {grand:>6,}")
print("=" * 55)
print(f"\n  Classes : {final_classes}")
print(f"  Target  : {TARGET} per class")
print(f"  Saved → {BALANCED_DIR}")
print(f"\n  ✔ Ready for multiclass fine-tuning!")

# DENSNET 4 CLASSES

In [ ]:
# =============================================================================
#  DenseNet169 Multiclass — FAST GPU VERSION (Kaggle P100 Optimized)
# =============================================================================

import os, random, warnings, pickle
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models
from torchvision.models import DenseNet169_Weights

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from tqdm import tqdm

warnings.filterwarnings("ignore")

# =============================================================================
# CONFIG
# =============================================================================

class CFG:
    DATA_DIR = "/kaggle/working/balanced_multiclass_final"

    SAVE_PATH = "/kaggle/working/best_model.pkl"

    IMG_SIZE = 224
    BATCH_SIZE = 64   # 🔥 increased for GPU
    EPOCHS = 20

    LR = 3e-4
    WEIGHT_DECAY = 1e-4

    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

torch.backends.cudnn.benchmark = True

# =============================================================================
# TRANSFORMS (OPTIMIZED)
# =============================================================================

train_tf = transforms.Compose([
    transforms.Resize((256,256)),
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(0.2,0.2,0.2,0.05),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],
                         [0.229,0.224,0.225]),
])

val_tf = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],
                         [0.229,0.224,0.225]),
])

# =============================================================================
# DATALOADER (FAST)
# =============================================================================

def build_loaders():
    dataset = datasets.ImageFolder(CFG.DATA_DIR)
    targets = np.array(dataset.targets)
    idx = np.arange(len(dataset))

    train_idx, temp_idx, _, temp_y = train_test_split(
        idx, targets, test_size=0.3, stratify=targets, random_state=42
    )

    val_idx, test_idx = train_test_split(
        temp_idx, test_size=0.5, stratify=temp_y, random_state=42
    )

    train_ds = datasets.ImageFolder(CFG.DATA_DIR, transform=train_tf)
    val_ds   = datasets.ImageFolder(CFG.DATA_DIR, transform=val_tf)
    test_ds  = datasets.ImageFolder(CFG.DATA_DIR, transform=val_tf)

    train_loader = DataLoader(
        Subset(train_ds, train_idx),
        batch_size=CFG.BATCH_SIZE,
        shuffle=True,
        num_workers=4,
        pin_memory=True,
        persistent_workers=True
    )

    val_loader = DataLoader(
        Subset(val_ds, val_idx),
        batch_size=CFG.BATCH_SIZE,
        shuffle=False,
        num_workers=4,
        pin_memory=True,
        persistent_workers=True
    )

    test_loader = DataLoader(
        Subset(test_ds, test_idx),
        batch_size=CFG.BATCH_SIZE,
        shuffle=False,
        num_workers=4,
        pin_memory=True,
        persistent_workers=True
    )

    return train_loader, val_loader, test_loader, dataset.classes

# =============================================================================
# MODEL
# =============================================================================

def build_model(num_classes):
    model = models.densenet169(weights=DenseNet169_Weights.IMAGENET1K_V1)

    in_f = model.classifier.in_features
    model.classifier = nn.Sequential(
        nn.Linear(in_f, 512),
        nn.ReLU(),
        nn.Dropout(0.5),
        nn.Linear(512, num_classes)
    )

    return model.to(CFG.DEVICE)

# =============================================================================
# TRAIN LOOP (AMP ENABLED)
# =============================================================================

def train(model, train_loader, val_loader, class_names):
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = optim.AdamW(model.parameters(), lr=CFG.LR)

    scaler = torch.cuda.amp.GradScaler()

    best_acc = 0

    for epoch in range(CFG.EPOCHS):
        print(f"\nEpoch {epoch+1}/{CFG.EPOCHS}")

        # ---------------- TRAIN ----------------
        model.train()
        total, correct = 0, 0

        for imgs, labels in tqdm(train_loader):
            imgs = imgs.to(CFG.DEVICE, non_blocking=True)
            labels = labels.to(CFG.DEVICE, non_blocking=True)

            optimizer.zero_grad()

            with torch.cuda.amp.autocast():
                outputs = model(imgs)
                loss = criterion(outputs, labels)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            preds = outputs.argmax(1)
            correct += (preds == labels).sum().item()
            total += imgs.size(0)

        train_acc = correct / total
        print(f"Train Acc: {train_acc:.4f}")

        # ---------------- VAL ----------------
        model.eval()
        total, correct = 0, 0

        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs = imgs.to(CFG.DEVICE, non_blocking=True)
                labels = labels.to(CFG.DEVICE, non_blocking=True)

                outputs = model(imgs)
                preds = outputs.argmax(1)

                correct += (preds == labels).sum().item()
                total += imgs.size(0)

        val_acc = correct / total
        print(f"Val Acc: {val_acc:.4f}")

        # SAVE BEST
        if val_acc > best_acc:
            best_acc = val_acc
            with open(CFG.SAVE_PATH, "wb") as f:
                pickle.dump(model.state_dict(), f)

            print("✔ Model Saved")

    print("\nBest Val Acc:", best_acc)

# =============================================================================
# TEST
# =============================================================================

def test(model, test_loader, class_names):
    with open(CFG.SAVE_PATH, "rb") as f:
        model.load_state_dict(pickle.load(f))

    model.eval()

    all_preds, all_labels = [], []

    with torch.no_grad():
        for imgs, labels in test_loader:
            imgs = imgs.to(CFG.DEVICE)
            outputs = model(imgs)
            preds = outputs.argmax(1).cpu().numpy()

            all_preds.extend(preds)
            all_labels.extend(labels.numpy())

    print("\nClassification Report:\n")
    print(classification_report(all_labels, all_preds, target_names=class_names))

    cm = confusion_matrix(all_labels, all_preds)

    sns.heatmap(cm, annot=True, fmt="d")
    plt.show()

# =============================================================================
# MAIN
# =============================================================================

def main():
    print("GPU:", torch.cuda.get_device_name(0))

    train_loader, val_loader, test_loader, class_names = build_loaders()

    model = build_model(len(class_names))

    train(model, train_loader, val_loader, class_names)

    test(model, test_loader, class_names)

if __name__ == "__main__":
    main()

In [ ]:
# =============================================================================
#  Multiclass Oral Cancer — DenseNet169 Fine-Tuning (GPU Optimized)
#  Classes  : Normal / OSCC / Benign / Leukoplakia
#  Dataset  : /kaggle/working/balanced_multiclass_final
#
#  Changes only for speed:
#  • AMP mixed precision
#  • cuDNN benchmark
#  • pin_memory + persistent_workers
#  • non_blocking GPU transfer
#  • channels_last for CNN speed
# =============================================================================

# ── Kaggle compatibility / reproducibility ───────────────────────────────────
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "0"

import io
import random
import warnings
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models
from torchvision.models import DenseNet169_Weights

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_curve,
    auc,
)
from tqdm import tqdm

warnings.filterwarnings("ignore")

torch.backends.cudnn.benchmark = True
torch.backends.cudnn.deterministic = False

# =============================================================================
#  MODULE 1 — CONFIG
# =============================================================================

class CFG:
    DATA_DIR  = "/kaggle/working/balanced_multiclass_final"

    # Fresh training: no old fine-tuned .pkl used
    PRETRAINED_PKL = None

    SAVE_PATH = "/kaggle/working/best_densenet169_multiclass.pkl"
    CKPT_PATH = "/kaggle/working/checkpoint_densenet169_multiclass.pkl"

    IMG_SIZE     = 224
    BATCH_SIZE   = 32
    WEIGHT_DECAY = 1e-4
    LABEL_SMOOTH = 0.10
    GRAD_CLIP    = 1.0
    PATIENCE     = 4

    # Phase 1 — train classifier head only
    LR_P1      = 3e-4
    EPOCHS_P1  = 5

    # Phase 2 — unfreeze deeper blocks, still conservative
    LR_P2      = 5e-5
    EPOCHS_P2  = 15

    TRAIN_RATIO = 0.70
    VAL_RATIO   = 0.15
    TEST_RATIO  = 0.15
    SEED        = 42

    T0      = 10
    T_MULT  = 2
    ETA_MIN = 1e-7

    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


def set_seed(seed=CFG.SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed()

print("=" * 72)
print("  DenseNet169 Multiclass — Normal / OSCC / Benign / Leukoplakia")
print(f"  Device      : {CFG.DEVICE}")
print(f"  Dataset     : {CFG.DATA_DIR}")
print(f"  Pretrained  : ImageNet only (fresh start)")
print(f"  PyTorch     : {torch.__version__}")
print("=" * 72, "\n")

if CFG.DEVICE == "cuda":
    try:
        _t = torch.zeros(2, 3, 4, 4).cuda()
        _conv = torch.nn.Conv2d(3, 3, 3, padding=1).cuda()
        _out = _conv(_t)
        print(f"  ✔ GPU sanity check passed  (shape: {_out.shape})\n")
        del _t, _conv, _out
    except Exception as e:
        print(f"  ✗ GPU check failed: {e}")
        print("  → Switching to CPU")
        CFG.DEVICE = "cpu"

# =============================================================================
#  MODULE 2 — TRANSFORMS
# =============================================================================

def get_transforms():
    train_tf = transforms.Compose([
        transforms.Resize((CFG.IMG_SIZE + 32, CFG.IMG_SIZE + 32)),
        transforms.RandomCrop(CFG.IMG_SIZE),
        transforms.RandomHorizontalFlip(p=0.50),
        transforms.RandomVerticalFlip(p=0.30),
        transforms.RandomRotation(degrees=20),
        transforms.ColorJitter(
            brightness=0.35, contrast=0.35,
            saturation=0.30, hue=0.10
        ),
        transforms.RandomAffine(
            degrees=0, translate=(0.10, 0.10),
            scale=(0.88, 1.12), shear=8
        ),
        transforms.RandomGrayscale(p=0.05),
        transforms.RandomPerspective(distortion_scale=0.25, p=0.30),
        transforms.ToTensor(),
        transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0)),
        transforms.Normalize([0.485, 0.456, 0.406],
                             [0.229, 0.224, 0.225]),
        transforms.RandomErasing(p=0.25, scale=(0.02, 0.12)),
    ])

    val_tf = transforms.Compose([
        transforms.Resize((CFG.IMG_SIZE, CFG.IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406],
                             [0.229, 0.224, 0.225]),
    ])
    return train_tf, val_tf

# =============================================================================
#  MODULE 3 — DATALOADERS (stratified 70 / 15 / 15)
# =============================================================================

def build_dataloaders(data_dir: str):
    train_tf, val_tf = get_transforms()

    base_ds = datasets.ImageFolder(root=data_dir)
    class_names = base_ds.classes
    targets = np.array(base_ds.targets)
    indices = np.arange(len(base_ds))

    print("▸ Dataset:")
    for i, c in enumerate(class_names):
        print(f"   [{i}] {c:<15}: {(targets == i).sum():>6,}")

    idx_train, idx_temp, _, y_temp = train_test_split(
        indices, targets,
        test_size=1 - CFG.TRAIN_RATIO,
        stratify=targets,
        random_state=CFG.SEED
    )

    val_frac = CFG.VAL_RATIO / (CFG.VAL_RATIO + CFG.TEST_RATIO)
    idx_val, idx_test = train_test_split(
        idx_temp,
        test_size=1 - val_frac,
        stratify=y_temp,
        random_state=CFG.SEED
    )

    print(f"\n▸ Split:")
    print(f"   Train : {len(idx_train):>6,}")
    print(f"   Val   : {len(idx_val):>6,}")
    print(f"   Test  : {len(idx_test):>6,}\n")

    train_ds = datasets.ImageFolder(root=data_dir, transform=train_tf)
    val_ds   = datasets.ImageFolder(root=data_dir, transform=val_tf)
    test_ds  = datasets.ImageFolder(root=data_dir, transform=val_tf)

    kw = dict(
        num_workers=4,
        pin_memory=True,
        persistent_workers=True,
        prefetch_factor=2,
    )

    train_loader = DataLoader(
        Subset(train_ds, idx_train),
        batch_size=CFG.BATCH_SIZE,
        shuffle=True,
        drop_last=True,
        **kw
    )
    val_loader = DataLoader(
        Subset(val_ds, idx_val),
        batch_size=CFG.BATCH_SIZE,
        shuffle=False,
        **kw
    )
    test_loader = DataLoader(
        Subset(test_ds, idx_test),
        batch_size=CFG.BATCH_SIZE,
        shuffle=False,
        **kw
    )

    return train_loader, val_loader, test_loader, class_names

# =============================================================================
#  MODULE 4 — MODEL
# =============================================================================

def build_model(num_classes: int = 4) -> nn.Module:
    model = models.densenet169(weights=DenseNet169_Weights.IMAGENET1K_V1)

    if CFG.PRETRAINED_PKL is not None and Path(CFG.PRETRAINED_PKL).exists():
        print("▸ PRETRAINED_PKL is set, but this fresh-training version ignores it.")
        print("  → Training starts from ImageNet weights only.\n")
    else:
        print("▸ Training from ImageNet weights only.\n")

    in_f = model.classifier.in_features
    model.classifier = nn.Sequential(
        nn.Linear(in_f, 512),
        nn.BatchNorm1d(512),
        nn.ReLU(inplace=True),
        nn.Dropout(p=0.50),

        nn.Linear(512, 128),
        nn.BatchNorm1d(128),
        nn.ReLU(inplace=True),
        nn.Dropout(p=0.40),

        nn.Linear(128, num_classes),
    )

    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

    print(f"▸ DenseNet169 ({num_classes} classes):")
    print(f"   Total    : {total:>12,}")
    print(f"   Trainable: {trainable:>12,}\n")

    model = model.to(CFG.DEVICE)
    model = model.to(memory_format=torch.channels_last)
    return model


def freeze_backbone(model):
    for param in model.parameters():
        param.requires_grad = False
    for param in model.classifier.parameters():
        param.requires_grad = True

    n = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"  Backbone FROZEN  |  trainable: {n:,}  (head only)")


def unfreeze_deep(model):
    for param in model.parameters():
        param.requires_grad = False

    unfreeze = False
    for name, module in model.features.named_children():
        if name == "denseblock3":
            unfreeze = True
        if unfreeze:
            for param in module.parameters():
                param.requires_grad = True

    for param in model.features.norm5.parameters():
        param.requires_grad = True
    for param in model.classifier.parameters():
        param.requires_grad = True

    n = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"  denseblock3-4 UNFROZEN  |  trainable: {n:,}")

# =============================================================================
#  MODULE 5 — CHECKPOINT
# =============================================================================

def save_best(model, optimizer, scheduler, epoch, val_acc, val_loss, class_names):
    with open(CFG.SAVE_PATH, "wb") as f:
        pickle.dump(model.state_dict(), f, protocol=pickle.HIGHEST_PROTOCOL)

    with open(CFG.CKPT_PATH, "wb") as f:
        pickle.dump({
            "epoch": epoch,
            "model_state": model.state_dict(),
            "optim_state": optimizer.state_dict(),
            "sched_state": scheduler.state_dict(),
            "val_acc": val_acc,
            "val_loss": val_loss,
            "class_names": class_names,
            "arch": "densenet169_multiclass_4class_fresh",
        }, f, protocol=pickle.HIGHEST_PROTOCOL)

    print(f"  💾 [epoch {epoch:02d}]  val_acc={val_acc*100:.2f}%  saved ✔")


def load_best(model):
    with open(CFG.SAVE_PATH, "rb") as f:
        state = pickle.load(f)
    model.load_state_dict(state)
    print(f"▸ Best weights loaded ← {CFG.SAVE_PATH}")
    return model

# =============================================================================
#  MODULE 6 — TRAIN / EVAL
# =============================================================================

def train_one_epoch(model, loader, criterion, optimizer, scaler):
    model.train()
    total_loss = correct = total = 0

    use_amp = (CFG.DEVICE == "cuda")

    for imgs, labels in tqdm(loader, desc="  Train", leave=False):
        imgs = imgs.to(CFG.DEVICE, non_blocking=True)
        labels = labels.to(CFG.DEVICE, non_blocking=True)
        imgs = imgs.contiguous(memory_format=torch.channels_last)

        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=use_amp):
            logits = model(imgs)
            loss = criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), CFG.GRAD_CLIP)
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item() * imgs.size(0)
        correct += (logits.argmax(1) == labels).sum().item()
        total += imgs.size(0)

    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss = correct = total = 0
    all_preds, all_labels, all_probs = [], [], []

    use_amp = (CFG.DEVICE == "cuda")

    for imgs, labels in tqdm(loader, desc="  Eval ", leave=False):
        imgs = imgs.to(CFG.DEVICE, non_blocking=True)
        labels = labels.to(CFG.DEVICE, non_blocking=True)
        imgs = imgs.contiguous(memory_format=torch.channels_last)

        with torch.cuda.amp.autocast(enabled=use_amp):
            logits = model(imgs)
            loss = criterion(logits, labels)
            probs = torch.softmax(logits, 1)
            preds = probs.argmax(1)

        total_loss += loss.item() * imgs.size(0)
        correct += (preds == labels).sum().item()
        total += imgs.size(0)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

    return total_loss / total, correct / total, all_preds, all_labels, all_probs

# =============================================================================
#  MODULE 7 — TWO-PHASE TRAINING LOOP
# =============================================================================

def run_phase(model, train_loader, val_loader, class_names,
              lr, max_epochs, phase_label,
              history, best_val_acc, best_val_loss):

    criterion = nn.CrossEntropyLoss(label_smoothing=CFG.LABEL_SMOOTH)
    optimizer = optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr,
        weight_decay=CFG.WEIGHT_DECAY
    )
    scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer,
        T_0=CFG.T0,
        T_mult=CFG.T_MULT,
        eta_min=CFG.ETA_MIN
    )
    scaler = torch.cuda.amp.GradScaler(enabled=(CFG.DEVICE == "cuda"))

    patience_ctr = 0
    global_ep = len(history["train_loss"])

    print(f"\n{'━' * 68}")
    print(f"  {phase_label}  |  lr={lr}  |  max {max_epochs} epochs")
    print(f"{'━' * 68}")

    for epoch in range(1, max_epochs + 1):
        global_ep += 1
        lr_now = optimizer.param_groups[0]["lr"]
        print(f"\nEpoch [{global_ep:02d}]  {phase_label}  lr={lr_now:.1e}")

        tr_loss, tr_acc = train_one_epoch(
            model, train_loader, criterion, optimizer, scaler
        )
        vl_loss, vl_acc, _, _, _ = evaluate(
            model, val_loader, criterion
        )
        scheduler.step()

        history["train_loss"].append(tr_loss)
        history["val_loss"].append(vl_loss)
        history["train_acc"].append(tr_acc)
        history["val_acc"].append(vl_acc)

        print(f"  Train → Loss:{tr_loss:.4f}  Acc:{tr_acc*100:.2f}%")
        print(f"  Val   → Loss:{vl_loss:.4f}  Acc:{vl_acc*100:.2f}%")

        improved = (vl_acc > best_val_acc or
                    (vl_acc == best_val_acc and vl_loss < best_val_loss))
        if improved:
            best_val_acc = vl_acc
            best_val_loss = vl_loss
            save_best(model, optimizer, scheduler,
                      global_ep, vl_acc, vl_loss, class_names)
            patience_ctr = 0
        else:
            patience_ctr += 1
            print(f"  ⚠ No improvement ({patience_ctr}/{CFG.PATIENCE})")
            if patience_ctr >= CFG.PATIENCE:
                print(f"\n⏹ Early stopping at epoch {global_ep}.")
                break

    return history, best_val_acc, best_val_loss


def train_model(model, train_loader, val_loader, class_names):
    history = {k: [] for k in ["train_loss", "val_loss", "train_acc", "val_acc"]}
    best_val_acc = 0.0
    best_val_loss = float("inf")

    freeze_backbone(model)
    history, best_val_acc, best_val_loss = run_phase(
        model, train_loader, val_loader, class_names,
        lr=CFG.LR_P1,
        max_epochs=CFG.EPOCHS_P1,
        phase_label="PHASE-1 Head Only",
        history=history,
        best_val_acc=best_val_acc,
        best_val_loss=best_val_loss
    )

    unfreeze_deep(model)
    history, best_val_acc, best_val_loss = run_phase(
        model, train_loader, val_loader, class_names,
        lr=CFG.LR_P2,
        max_epochs=CFG.EPOCHS_P2,
        phase_label="PHASE-2 Deep Fine-tune",
        history=history,
        best_val_acc=best_val_acc,
        best_val_loss=best_val_loss
    )

    print(f"\n{'─' * 68}")
    print(f"  Training Complete | Best Val Acc: {best_val_acc*100:.2f}%")
    print(f"{'─' * 68}\n")
    return history

# =============================================================================
#  MODULE 8 — VISUALISATIONS
# =============================================================================

def plot_training_curves(history):
    ep = range(1, len(history["train_loss"]) + 1)
    p1_end = CFG.EPOCHS_P1

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    axes[0].plot(ep, history["train_loss"], "b-o", ms=4, lw=1.8, label="Train")
    axes[0].plot(ep, history["val_loss"],   "r-s", ms=4, lw=1.8, label="Val")
    axes[0].axvline(p1_end, color="orange", linestyle="--",
                    lw=1.5, label="Phase 1→2")
    axes[0].set_title("Loss Curve", fontweight="bold")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")
    axes[0].legend()
    axes[0].grid(alpha=0.35)
    axes[0].spines[["top", "right"]].set_visible(False)

    tr_pct = [a * 100 for a in history["train_acc"]]
    vl_pct = [a * 100 for a in history["val_acc"]]
    axes[1].plot(ep, tr_pct, "b-o", ms=4, lw=1.8, label="Train Acc")
    axes[1].plot(ep, vl_pct, "r-s", ms=4, lw=1.8, label="Val Acc")
    axes[1].axhline(max(vl_pct), color="green", linestyle=":",
                    lw=1.5, label=f"Best {max(vl_pct):.1f}%")
    axes[1].axvline(p1_end, color="orange", linestyle="--",
                    lw=1.5, label="Phase 1→2")
    axes[1].set_title("Accuracy Curve", fontweight="bold")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Accuracy (%)")
    axes[1].legend()
    axes[1].grid(alpha=0.35)
    axes[1].spines[["top", "right"]].set_visible(False)

    plt.suptitle("DenseNet169 Multiclass — Normal / OSCC / Benign / Leukoplakia",
                 fontsize=14, fontweight="bold")
    plt.tight_layout()
    out = "/kaggle/working/multiclass_training_curves.png"
    plt.savefig(out, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close()
    print(f"▸ Saved: {out}")


def plot_confusion_matrix(cm, class_names):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for ax, data, fmt, cmap, title in zip(
        axes,
        [cm, cm.astype(float) / cm.sum(axis=1, keepdims=True)],
        ["d", ".2%"],
        ["Blues", "Greens"],
        ["Counts", "Normalised"]
    ):
        sns.heatmap(
            data, annot=True, fmt=fmt, cmap=cmap,
            xticklabels=class_names, yticklabels=class_names,
            linewidths=0.8, ax=ax,
            annot_kws={"size": 13, "weight": "bold"}
        )
        ax.set_xlabel("Predicted")
        ax.set_ylabel("Actual")
        ax.set_title(f"Confusion Matrix ({title})", fontweight="bold")

    plt.suptitle("DenseNet169 Multiclass — Confusion Matrix",
                 fontsize=14, fontweight="bold")
    plt.tight_layout()
    out = "/kaggle/working/multiclass_confusion_matrix.png"
    plt.savefig(out, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close()
    print(f"▸ Saved: {out}")


def plot_per_class_metrics(per_class, class_names):
    metrics = ["Precision", "Recall", "F1-Score"]
    keys = ["precision", "recall", "f1score"]
    colors = ["#4C72B0", "#DD8452", "#55A868"]
    x, w = np.arange(len(class_names)), 0.25

    fig, ax = plt.subplots(figsize=(12, 5))
    for i, (m, k, c) in enumerate(zip(metrics, keys, colors)):
        vals = [per_class[cls][k] for cls in class_names]
        bars = ax.bar(x + i * w, vals, w, label=m, color=c,
                      alpha=0.88, edgecolor="white")
        for bar, v in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width() / 2,
                    bar.get_height() + 0.012, f"{v:.3f}",
                    ha="center", va="bottom",
                    fontsize=9, fontweight="bold")

    ax.set_xticks(x + w)
    ax.set_xticklabels(class_names, fontsize=12)
    ax.set_ylim(0, 1.18)
    ax.set_ylabel("Score")
    ax.set_title("Per-Class Precision / Recall / F1",
                 fontsize=13, fontweight="bold")
    ax.legend()
    ax.grid(axis="y", alpha=0.35)
    ax.spines[["top", "right"]].set_visible(False)
    plt.tight_layout()
    out = "/kaggle/working/multiclass_per_class_metrics.png"
    plt.savefig(out, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close()
    print(f"▸ Saved: {out}")


def plot_roc_curves(model, test_loader, class_names):
    model.eval()
    all_probs, all_labels = [], []

    with torch.no_grad():
        for imgs, labels in test_loader:
            imgs = imgs.to(CFG.DEVICE, non_blocking=True)
            imgs = imgs.contiguous(memory_format=torch.channels_last)
            probs = torch.softmax(model(imgs), 1).cpu().numpy()
            all_probs.extend(probs)
            all_labels.extend(labels.numpy())

    all_probs = np.array(all_probs)
    all_labels = np.array(all_labels)

    colors = plt.cm.tab10(np.linspace(0, 1, len(class_names)))

    fig, ax = plt.subplots(figsize=(8, 6))
    for i, (cls, col) in enumerate(zip(class_names, colors)):
        y_true = (all_labels == i).astype(int)
        y_score = all_probs[:, i]
        fpr, tpr, _ = roc_curve(y_true, y_score)
        roc_auc = auc(fpr, tpr)
        ax.plot(fpr, tpr, color=col, lw=2, label=f"{cls}  (AUC={roc_auc:.3f})")

    ax.plot([0, 1], [0, 1], "k--", lw=1.2)
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_title("ROC Curves — One-vs-Rest", fontweight="bold")
    ax.legend()
    ax.grid(alpha=0.35)
    ax.spines[["top", "right"]].set_visible(False)
    plt.tight_layout()
    out = "/kaggle/working/multiclass_roc_curves.png"
    plt.savefig(out, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close()
    print(f"▸ Saved: {out}")


@torch.no_grad()
def visualize_predictions(model, test_loader, class_names, n=12):
    model.eval()
    all_imgs, all_labels = [], []

    for imgs, labels in test_loader:
        all_imgs.append(imgs)
        all_labels.append(labels)
        if sum(len(x) for x in all_imgs) >= n:
            break

    imgs_cat = torch.cat(all_imgs)[:n]
    labels_cat = torch.cat(all_labels)[:n]
    imgs_cat = imgs_cat.to(CFG.DEVICE, non_blocking=True)
    imgs_cat = imgs_cat.contiguous(memory_format=torch.channels_last)

    probs = torch.softmax(model(imgs_cat), 1).cpu()
    preds = probs.argmax(1)

    mean_ = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std_  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    imgs_show = (torch.cat(all_imgs)[:n] * std_ + mean_).clamp(0, 1)

    rows = 3
    cols = 4
    fig, axes = plt.subplots(rows, cols, figsize=(16, 12))
    for i, ax in enumerate(axes.flat):
        if i >= n:
            ax.axis("off")
            continue

        ax.imshow(imgs_show[i].permute(1, 2, 0).numpy())
        pred_lbl = class_names[preds[i]]
        true_lbl = class_names[labels_cat[i]]
        conf = probs[i][preds[i]].item() * 100
        ok = pred_lbl == true_lbl
        color = "#2ecc71" if ok else "#e74c3c"

        ax.set_title(
            f"Pred:{pred_lbl}\nTrue:{true_lbl}\n{conf:.1f}%",
            color=color, fontsize=9, fontweight="bold"
        )
        for s in ax.spines.values():
            s.set_visible(True)
            s.set_edgecolor(color)
            s.set_linewidth(3)
        ax.set_xticks([])
        ax.set_yticks([])

    cp = mpatches.Patch(color="#2ecc71", label="Correct")
    wp = mpatches.Patch(color="#e74c3c", label="Wrong")
    fig.legend(handles=[cp, wp], loc="lower center",
               ncol=2, fontsize=11, bbox_to_anchor=(0.5, 0.0))
    plt.suptitle("DenseNet169 Multiclass — Sample Predictions",
                 fontsize=14, fontweight="bold")
    plt.tight_layout()
    out = "/kaggle/working/multiclass_sample_predictions.png"
    plt.savefig(out, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close()
    print(f"▸ Saved: {out}")

# =============================================================================
#  MODULE 9 — TEST EVALUATION
# =============================================================================

def evaluate_test(model, test_loader, class_names):
    model = load_best(model)
    criterion = nn.CrossEntropyLoss()

    te_loss, te_acc, preds, labels, probs = evaluate(
        model, test_loader, criterion
    )

    prec_w = precision_score(labels, preds, average="weighted", zero_division=0)
    rec_w  = recall_score(labels, preds, average="weighted", zero_division=0)
    f1_w   = f1_score(labels, preds, average="weighted", zero_division=0)
    cm     = confusion_matrix(labels, preds)

    prec_pc = precision_score(labels, preds, average=None, zero_division=0)
    rec_pc  = recall_score(labels, preds, average=None, zero_division=0)
    f1_pc   = f1_score(labels, preds, average=None, zero_division=0)

    per_class = {
        cls: {
            "precision": prec_pc[i],
            "recall":    rec_pc[i],
            "f1score":   f1_pc[i]
        }
        for i, cls in enumerate(class_names)
    }

    print(f"\n{'═' * 68}")
    print(f"  TEST RESULTS — DenseNet169 Multiclass")
    print(f"{'═' * 68}")
    print(f"  Accuracy  : {te_acc*100:.4f}%")
    print(f"  Loss      : {te_loss:.4f}")
    print(f"  Precision : {prec_w:.4f}  (weighted)")
    print(f"  Recall    : {rec_w:.4f}  (weighted)")
    print(f"  F1-Score  : {f1_w:.4f}  (weighted)")

    print(f"\n  Per-Class:")
    print(f"  {'Class':<15} {'P':>8} {'R':>8} {'F1':>8}")
    print(f"  {'─' * 42}")
    for cls, m in per_class.items():
        print(f"  {cls:<15} {m['precision']:>8.4f} {m['recall']:>8.4f} {m['f1score']:>8.4f}")

    print(f"\n{classification_report(labels, preds, target_names=class_names, digits=4)}")

    rows = [{
        "Class": "OVERALL",
        "Precision": prec_w,
        "Recall": rec_w,
        "F1": f1_w,
        "Accuracy": te_acc
    }]
    for cls, m in per_class.items():
        rows.append({
            "Class": cls,
            "Precision": m["precision"],
            "Recall": m["recall"],
            "F1": m["f1score"],
            "Accuracy": ""
        })

    pd.DataFrame(rows).to_csv(
        "/kaggle/working/multiclass_test_metrics.csv",
        index=False
    )
    print("▸ Saved: multiclass_test_metrics.csv")

    plot_confusion_matrix(cm, class_names)
    plot_per_class_metrics(per_class, class_names)
    plot_roc_curves(model, test_loader, class_names)
    visualize_predictions(model, test_loader, class_names)

    return {
        "accuracy": te_acc,
        "precision": prec_w,
        "recall": rec_w,
        "f1": f1_w,
        "per_class": per_class
    }

# =============================================================================
#  MODULE 10 — MAIN
# =============================================================================

def main():
    train_loader, val_loader, test_loader, class_names = build_dataloaders(CFG.DATA_DIR)

    model = build_model(num_classes=len(class_names))

    history = train_model(model, train_loader, val_loader, class_names)

    plot_training_curves(history)

    metrics = evaluate_test(model, test_loader, class_names)

    print(f"\n{'═' * 68}")
    print(f"  ✅  ALL DONE — DenseNet169 Multiclass (GPU optimized)")
    print(f"{'═' * 68}")
    print(f"  Accuracy  : {metrics['accuracy']*100:.2f}%")
    print(f"  F1-Score  : {metrics['f1']:.4f}")
    print(f"\n  Outputs → /kaggle/working/")
    for f in [
        "best_densenet169_multiclass.pkl",
        "checkpoint_densenet169_multiclass.pkl",
        "multiclass_training_curves.png",
        "multiclass_confusion_matrix.png",
        "multiclass_per_class_metrics.png",
        "multiclass_roc_curves.png",
        "multiclass_sample_predictions.png",
        "multiclass_test_metrics.csv",
    ]:
        print(f"  • {f}")
    print(f"{'═' * 68}\n")

    return metrics


if __name__ == "__main__":
    main()

In [ ]:
import shutil

shutil.copy("/kaggle/working/model.zip", "/kaggle/working/model_download.zip")

In [ ]:
import base64

file_path = "/kaggle/working/best_densenet169_multiclass.pkl"

with open(file_path, "rb") as f:
    data = f.read()

b64 = base64.b64encode(data).decode()

from IPython.display import HTML

HTML(f'''
<a download="best_model.pkl" href="data:application/octet-stream;base64,{b64}">
    👉 CLICK HERE TO DOWNLOAD MODEL
</a>
''')

In [ ]:
pip install numpy==1.26.4 -q

In [ ]:
import os, glob, sys

print("CWD:", os.getcwd())
print("torch-like files/folders:", glob.glob("torch*"))
print("python path head:", sys.path[:5])

In [ ]:
!pip uninstall -y torch torchvision torchaudio -q
!pip install --no-cache-dir torch==2.2.2+cu121 torchvision==0.17.2+cu121 torchaudio==2.2.2+cu121 --index-url https://download.pytorch.org/whl/cu121 -q

In [ ]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))